# 实践 1.2  文件校验、样本映射与批量任务表

《统计遗传与智能育种学》第一章 · 电子资源 `01b_workflow_manifest.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：01b_workflow_manifest.ipynb。实践不依赖真实FASTQ，而是生成多个小型文本数据文件，计算SHA-256校验值，构建样本—文件manifest，预设一个样本命名错误并通过集合差异定位问题；随后根据manifest生成可审计的批量任务表。该练习用于训练样本标识、文件完整性校验、工作流组织和可追溯记录等基础数据管理能力。

运行检查：重启内核并运行全部单元  
预期：输出文件校验表、重复/缺失样本检查、批量任务manifest；能够定位预设的样本ID错误。

## 输入、完整代码、实际输出与解释

输入与可复现条件：5个预期样本的小型文本文件；预先将S03文件命名为SX3；SEED=102；运行会生成outputs_01_manifest目录中的示例输入。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=102
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 102


In [2]:
from pathlib import Path
import hashlib
root=Path('outputs_01_manifest'); root.mkdir(exist_ok=True)
expected=[f'S{i:02d}' for i in range(1,6)]
for sid in expected:
    (root/f'{sid}.txt').write_text(f'sample_id\tvalue\n{sid}\t{rng.integers(10,99)}\n',encoding='utf-8')
# 制造一个命名错误：文件内容仍是S03，但文件名改成SX3
(root/'S03.txt').rename(root/'SX3.txt')
def sha256(path):
    h=hashlib.sha256(); h.update(path.read_bytes()); return h.hexdigest()
files=sorted(root.glob('*.txt'))
manifest=pd.DataFrame({'file':[p.name for p in files],'sample_id':[p.stem for p in files],'sha256':[sha256(p) for p in files]})
print(manifest.to_string(index=False))

observed=set(manifest.sample_id); expected_set=set(expected)
print('expected but missing:',sorted(expected_set-observed))
print('unexpected names:',sorted(observed-expected_set))
# 修复命名后重新生成任务表
(root/'SX3.txt').rename(root/'S03.txt')
files=sorted(root.glob('*.txt'))
tasks=pd.DataFrame({'sample_id':[p.stem for p in files],'input':[str(p) for p in files]})
tasks['output']=tasks.sample_id.map(lambda s:f'results/{s}.result.tsv')
tasks['command']=tasks.apply(lambda r:f"analyze --input {r['input']} --output {r['output']}",axis=1)
print(tasks.to_string(index=False))


   file sample_id                                                           sha256
S01.txt       S01 4f03ed1bda139633d721751d18acb4ce66cd8625e49ea895103dc3eee172a35a
S02.txt       S02 1f805e43339080a0e17d8b7feef4faacefba89d10cba19764ef528e1cb57cd8b
S04.txt       S04 1e9f90a5aabc0e0e448f1728e95bdaebf7adec6ce193d349d645fb6c0c21d4b1
S05.txt       S05 e616299a0ca033340b8c4896fa687613cb295e45b8a25e9382fb35f351117a8b
SX3.txt       SX3 ba1104c737211b630b514f9228a641dc78768fe19c97d700e9fa5e230d3f1348
expected but missing: ['S03']
unexpected names: ['SX3']
sample_id                       input                 output                                                                     command
      S01 outputs_01_manifest/S01.txt results/S01.result.tsv analyze --input outputs_01_manifest/S01.txt --output results/S01.result.tsv
      S02 outputs_01_manifest/S02.txt results/S02.result.tsv analyze --input outputs_01_manifest/S02.txt --output results/S02.result.tsv
      S03 outputs_01_manifest/S03.t

结果解释：集合核对准确报告“缺S03、多SX3”，SHA-256为每个文件提供字节级指纹；校验值能发现文件内容变化，但不能证明生物学样本身份正确。批量任务表由修改后的文件清单生成，可审计每个输入、输出和命令。

可修改练习：修改S02.txt中的一个数值并重算校验值；再把S03改名为SX3，比较“内容改变”和“样本命名错误”分别由哪一步发现。